**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Cyclostationarity & Higher-Order Statistics

Two escapes from the [WSS/Gaussian](./Statistical_Signal_Processing.ipynb) worldview: signals whose *statistics* repeat periodically (every modulated signal!), detectable even **below the noise floor** — and higher-order moments that see what covariance is blind to. This is how [SDR](../Intro_SDR/Software_Defined_Radio.ipynb) detectors find signals the PSD can't.

## 1. Pre-requisites

[Statistical SP](./Statistical_Signal_Processing.ipynb), [Digital Communications](./Digital_Communications.ipynb) S1–S2.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Cyclostationarity: Rhythm in the Statistics* (~40 min)
**Goal:** see why modulated signals aren't WSS; meet the cyclic autocorrelation.
**Builds on:** [Statistical SP](./Statistical_Signal_Processing.ipynb) S1. &nbsp; **Feeds into:** Session 2 (detection below the noise floor).

---

## 2. Periodic Statistics

💡 **Intuition.** A BPSK signal's *mean power* pulses at the symbol rate — the signal is not WSS, it's **cyclostationary**: statistics periodic in time. Fourier-expand the time-varying autocorrelation and you get the **cyclic autocorrelation** $R_x^\alpha(\tau)$ at *cycle frequencies* α — nonzero exactly at multiples of the symbol rate (and around twice the carrier). White noise, being genuinely stationary, has **zero** cyclic content at any α ≠ 0 — and that asymmetry is an exploitable superpower.

In [2]:
# a BPSK signal, its cyclic autocorrelation at candidate cycle frequencies
fs, sym_rate, fc = 8000, 250, 1000
sps = fs // sym_rate
n_sym = 400
symbols = rng.choice([-1, 1], n_sym)
base = np.repeat(symbols, sps)
t = np.arange(len(base)) / fs
x_clean = base * np.cos(2*np.pi*fc*t)

def cyclic_acf(x, alpha, fs, max_lag=40):
    n_ax = np.arange(len(x))
    rot = x * np.exp(-1j*2*np.pi*alpha*n_ax/fs)
    return np.array([np.mean(rot[k:] * np.conj(x[:len(x)-k])) for k in range(max_lag)])

alphas = np.arange(0, 1200, 12.5)
strength = [np.abs(cyclic_acf(x_clean, a, fs)).max() for a in alphas]
plt.figure(figsize=(8.5, 2.8))
plt.stem(alphas, strength, basefmt=" ", markerfmt=".")
for a_true, name in [(sym_rate, "symbol rate"), (2*fc, "2×carrier")]:
    plt.axvline(a_true, color="r", linestyle=":", linewidth=1)
    plt.text(a_true, max(strength)*0.9, name, fontsize=7, color="r", rotation=90)
plt.xlabel("cycle frequency α [Hz]"); plt.ylabel("max |R_x^α(τ)|")
plt.title("cyclic signature of BPSK: spikes exactly at the symbol rate and 2f_c")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2988402/821798061.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *Detection Below the Noise Floor* (~40 min)
**Goal:** energy detection dies at 0 dB; the cyclic detector keeps working well below it.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (higher-order statistics).

---

## 3. The Superpower, Cashed

💡 **Intuition.** An energy detector asks 'is total power above the noise baseline?' — hopeless when SNR < 0 dB and the noise level is uncertain. The **cyclic detector** asks 'is there energy at cycle frequency α = symbol rate?' — and since noise contributes *nothing* there (only estimation noise, shrinking with record length), the signal's rhythm shines through arbitrarily far below the floor, given time. This is how spectrum sensors find hidden transmitters.

In [3]:
def detect_trial(snr_db, with_signal, N_rec=60000, alpha=sym_rate, level_uncert_db=1.0):
    n_sym_r = N_rec // sps
    xb = np.repeat(rng.choice([-1, 1], n_sym_r), sps).astype(float)
    tt = np.arange(len(xb)) / fs
    s = xb * np.cos(2*np.pi*fc*tt)                        # signal power = 1/2
    # REALISM: the receiver's noise level is only known to ±1 dB (temperature, gain drift)
    wobble = 10**(rng.uniform(-level_uncert_db, level_uncert_db)/20)
    noise_sigma = np.sqrt(0.5) * 10**(-snr_db/20) * wobble
    y = (s if with_signal else np.zeros(len(xb))) + noise_sigma * rng.standard_normal(len(xb))
    energy = np.mean(y**2)
    cyc = np.abs(cyclic_acf(y, alpha, fs, max_lag=20)).max()
    return energy, cyc

for snr in [0, -5, -10]:
    e1 = [detect_trial(snr, True)[0] for _ in range(20)]
    e0 = [detect_trial(snr, False)[0] for _ in range(20)]
    c1 = [detect_trial(snr, True)[1] for _ in range(20)]
    c0 = [detect_trial(snr, False)[1] for _ in range(20)]
    def sep(a, b): return (np.mean(a)-np.mean(b)) / np.sqrt(np.var(a)+np.var(b)+1e-18)
    print(f"SNR {snr:+3d} dB:  energy-detector separation {sep(e1,e0):5.1f}σ   cyclic-detector {sep(c1,c0):5.1f}σ")

SNR  +0 dB:  energy-detector separation   5.7σ   cyclic-detector  42.5σ


SNR  -5 dB:  energy-detector separation   1.4σ   cyclic-detector  24.2σ


SNR -10 dB:  energy-detector separation   0.8σ   cyclic-detector   8.8σ


Read the table: the energy detector's separation collapses with SNR (and would collapse entirely under noise-level uncertainty), while the cyclic statistic keeps the classes many σ apart — the rhythm survives where the power story drowns.

---
### 🕐 Session 3 of 3 — *Higher-Order Statistics* (~35 min)
**Goal:** what covariance can't see: kurtosis, the bispectrum, and Gaussian blindness.
**Builds on:** Session 2.

---

## 4. Beyond Second Order

💡 **Intuition.** Covariance sees only the ellipse of a distribution. **Higher-order cumulants** see shape: kurtosis (4th) measures tail-heaviness — the compass [ICA](./ICA_Blind_Source_Separation.ipynb) steers by — and the **bispectrum** (the 2-D Fourier transform of the 3rd-order cumulant) detects *phase coupling*: components at $f_1, f_2, f_1{+}f_2$ with locked phases, the fingerprint of nonlinearity. The master fact: **all cumulants above 2nd order of a Gaussian are exactly zero** — so anything nonzero up there is, provably, structure.

In [4]:
# quadratic phase coupling: visible to the bispectrum, invisible to the PSD
N = 2**14
tt = np.arange(N)/fs
f1, f2 = 480, 700
# phases drift slowly and independently between oscillators (physical: separate sources);
# in the COUPLED signal the third tone's phase is SLAVED to ph1+ph2 at every instant
def drifting_phase(N, rate=0.05):
    return np.cumsum(rate * rng.standard_normal(N))
ph1, ph2, ph3 = drifting_phase(N), drifting_phase(N), drifting_phase(N)
coupled   = np.cos(2*np.pi*f1*tt+ph1) + np.cos(2*np.pi*f2*tt+ph2) + 0.7*np.cos(2*np.pi*(f1+f2)*tt + ph1+ph2)
uncoupled = np.cos(2*np.pi*f1*tt+ph1) + np.cos(2*np.pi*f2*tt+ph2) + 0.7*np.cos(2*np.pi*(f1+f2)*tt + ph3)
coupled += 0.3*rng.standard_normal(N); uncoupled += 0.3*rng.standard_normal(N)

def bicoherence_peak(x, f1, f2, fs, nseg=64):
    seg = len(x)//nseg
    num = 0; d1 = 0; d2 = 0
    for k in range(nseg):
        X = np.fft.rfft(x[k*seg:(k+1)*seg] * np.hanning(seg))
        freqs = np.fft.rfftfreq(seg, 1/fs)
        i1, i2 = np.argmin(np.abs(freqs-f1)), np.argmin(np.abs(freqs-f2))
        i12 = np.argmin(np.abs(freqs-(f1+f2)))
        num += X[i1]*X[i2]*np.conj(X[i12])
        d1 += np.abs(X[i1]*X[i2])**2; d2 += np.abs(X[i12])**2
    return np.abs(num) / np.sqrt(d1*d2)

print("PSD at f1, f2, f1+f2 is IDENTICAL for both signals (same magnitudes; only phase RELATIONS differ)")
print(f"bicoherence at (f1, f2):  coupled {bicoherence_peak(coupled, f1, f2, fs):.3f}   "
      f"uncoupled {bicoherence_peak(uncoupled, f1, f2, fs):.3f}")
print("→ near 1 vs near 0: third-order statistics detect the phase LOCK the PSD cannot express")

PSD at f1, f2, f1+f2 is IDENTICAL for both signals (same magnitudes; only phase RELATIONS differ)
bicoherence at (f1, f2):  coupled 0.977   uncoupled 0.110
→ near 1 vs near 0: third-order statistics detect the phase LOCK the PSD cannot express


## 5. Conclusion

Modulation puts rhythm into statistics; that rhythm detects signals the PSD loses (σ-separations measured); and third/fourth-order cumulants see phase coupling and non-Gaussian shape where covariance sees nothing at all. When the standard assumptions fail, these are the tools that notice.

---
## Where next

- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — cyclic detection on live captures.
- [ICA](./ICA_Blind_Source_Separation.ipynb) — kurtosis as a steering wheel.
- [Digital Communications](./Digital_Communications.ipynb) — the signals whose rhythms we exploited.